# Dense WDM runtime optimisation benchmark

This notebook is the controlled performance laboratory for the full-year, six-link WDM one-arm response. It deliberately does **not** replace the scientific validation notebooks.

The frozen physical problem is:

- $N=3{,}200{,}000$, $dt=10\,\mathrm{s}$, $N_f=100$;
- `calc_m0=False`, $L_{\rm trunc}=25$;
- production `complex64` CPU assembly;
- current reference tuning: `row_chunk_size=32640`, `lag_block_size=17`;
- current link schedule: groups of 2 links on 3 workers.

Reference measurements on `calx038` were approximately 0.124 s for the paired DWT stage, 0.328 s for the selected six-link shift, and 0.453 s for complete A1. Treat those values as a sanity check, not hard pass/fail thresholds.

Run this notebook from a freshly restarted kernel. It tunes one dimension at a time, checks every output against a frozen link map, and exports JSON/CSV records. Only winners should proceed to the existing end-to-end response/mismatch validation.

## 0. CPU environment and imports

The backend variables must be set before importing JAX, FEW, WDM, or the response package.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""


In [2]:
import gc
import json
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in [ROOT, *ROOT.parents]
        if (path / "WDM_LISAresponse").exists()
        and (path / "WDM_GW_wavelets").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Run from a directory beneath a parent containing both "
        "WDM_LISAresponse and WDM_GW_wavelets."
    )

WDM_LISA_ROOT = PROJECT_ROOT / "WDM_LISAresponse"
WDM_GW_ROOT = PROJECT_ROOT / "WDM_GW_wavelets"
BENCHMARK_DIR = WDM_LISA_ROOT / "testing_optimisation"

for source_path in (
    WDM_LISA_ROOT / "src",
    WDM_GW_ROOT / "src",
    BENCHMARK_DIR,
):
    source_text = str(source_path)
    if source_text not in sys.path:
        sys.path.insert(0, source_text)

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import WDM
import WDM_LISAresponse
from few.waveform import GenerateEMRIWaveform
from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

from benchmark_runtime_helpers import (
    benchmark_candidate,
    block_result,
    compare_arrays,
    compare_link_maps,
    fastest_passing,
    git_state,
    memory_snapshot,
    numerical_gate,
    save_json,
    system_metadata,
    timed_call,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMStaticPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_context_from_wdm,
    _build_wdm_dT_grid,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

if jax.default_backend() != "cpu":
    raise RuntimeError(f"Expected the CPU backend, got {jax.default_backend()!r}.")


## 1. Experiment controls

`smoke` verifies that the notebook works, `quick` is suitable while developing, and `full` is the result-producing run. The sweep is intentionally staged rather than Cartesian: row size first, then lag block size at the winning row size, then link scheduling on the winning kernel configuration.

In [3]:
RUN_MODE = "full"  # "smoke", "quick", or "full"

RUN_PROFILES = {
    "smoke": {
        "warmups": 0,
        "repeats": 1,
        "a1_repeats": 1,
        "row_sizes": [32_000, 32_640],
        "lag_sizes": [17, 51],
    },
    "quick": {
        "warmups": 1,
        "repeats": 5,
        "a1_repeats": 3,
        "row_sizes": [8_192, 16_000, 32_000, 32_640],
        "lag_sizes": [8, 16, 17, 25],
    },
    "full": {
        "warmups": 2,
        "repeats": 11,
        "a1_repeats": 9,
        "row_sizes": [4_096, 8_192, 16_000, 24_000, 32_000, 32_640],
        "lag_sizes": [3, 8, 16, 17, 25, 51],
    },
}
PROFILE = RUN_PROFILES[RUN_MODE]

N_UNPADDED = 3_200_000
DT = 10.0
NF = 100
Q_WDM = 4
D_WDM = 4
A_FRAC = 0.25
WINDOW_ALPHA = 0.005
TUKEY_FACTOR = 4.0
L_TRUNC = 25
CALC_M0 = False

# Match generate_windowed_padded_waveform:
# symmetric Tukey padding, followed by optional right-tail padding
# to a multiple of 2*Nf.
WINDOW_PAD_PER_SIDE = int(
    np.ceil(TUKEY_FACTOR * WINDOW_ALPHA * 0.5 * N_UNPADDED)
)

POST_WINDOW_N = N_UNPADDED + 2 * WINDOW_PAD_PER_SIDE
WDM_LENGTH_MULTIPLE = 2 * NF

EXPECTED_PADDED_N = (
    (POST_WINDOW_N + WDM_LENGTH_MULTIPLE - 1)
    // WDM_LENGTH_MULTIPLE
) * WDM_LENGTH_MULTIPLE

EXPECTED_WDM_NT = EXPECTED_PADDED_N // NF

print("Unpadded samples:", N_UNPADDED)
print("Padding per side:", WINDOW_PAD_PER_SIDE)
print("Expected padded samples:", EXPECTED_PADDED_N)
print("Expected WDM Nt:", EXPECTED_WDM_NT)

BASE_ROW_CHUNK_SIZE = 32_640
BASE_LAG_BLOCK_SIZE = 17
BASE_GROUP_SIZE = 2
BASE_WORKERS = 3

BASE_SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": BASE_ROW_CHUNK_SIZE,
    "lag_block_size": BASE_LAG_BLOCK_SIZE,
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

SCHEDULES = [
    {
        "name": "runtime_sky",
        "mode": "runtime_sky",
        "group_size": 6,
        "workers": 1,
    },
]

REL_L2_TOLERANCE = 1.0e-6
MAX_ABS_TOLERANCE = None
MIN_MEANINGFUL_SPEEDUP = 1.05

REUSE_WAVEFORM_CACHE = True
REUSE_WDM_CACHE = True
RUN_DWT_BENCHMARK = True
RUN_COMPLETE_A1 = True
SAVE_RESULTS = True
STRICT_PROVENANCE = False
EXPECTED_WDM_COMMIT_PREFIX = "c588649f5"

# Add one entry after implementing a new exact backend or feature switch.
# It will be compared against the same frozen link output, but it is not
# automatically promoted into the staged row/lag/schedule winner.
CUSTOM_EXPERIMENTS = []


Unpadded samples: 3200000
Padding per side: 32000
Expected padded samples: 3264000
Expected WDM Nt: 32640


## 2. Provenance and fixed physical inputs

A dirty checkout is recorded rather than hidden. Set `STRICT_PROVENANCE=True` only after the intended WDM commit is pinned and the worktrees are clean.

In [4]:
HOST = system_metadata()
WDM_STATE = git_state(WDM.__file__)
RESPONSE_STATE = git_state(WDM_LISAresponse.__file__)

print(json.dumps({"host": HOST, "WDM": WDM_STATE, "response": RESPONSE_STATE}, indent=2))

wdm_commit = WDM_STATE.get("commit") or ""
provenance_ok = wdm_commit.startswith(EXPECTED_WDM_COMMIT_PREFIX)
if not provenance_ok:
    message = (
        f"WDM commit {wdm_commit or 'unknown'} does not match the expected "
        f"prefix {EXPECTED_WDM_COMMIT_PREFIX!r}."
    )
    if STRICT_PROVENANCE:
        raise RuntimeError(message)
    warnings.warn(message)

if WDM_STATE.get("dirty") or RESPONSE_STATE.get("dirty"):
    message = "At least one benchmark checkout is dirty; the JSON record will preserve this fact."
    if STRICT_PROVENANCE:
        raise RuntimeError(message)
    warnings.warn(message)

ORBIT_FILE = WDM_LISA_ROOT / "examples" / "esa-trailing-orbits.h5"
if not ORBIT_FILE.exists():
    raise FileNotFoundError(ORBIT_FILE)

orbit_data = read_orbit_file(str(ORBIT_FILE))
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

THETA = np.pi / 2
PHI = 0.0
print("Initial memory:", memory_snapshot())


{
  "host": {
    "node": "calx038.ast.cam.ac.uk",
    "platform": "Linux-5.14.0-687.30.1.el9_8.x86_64-x86_64-with-glibc2.34",
    "processor": "x86_64",
    "python": "3.12.12",
    "jax": "0.9.2",
    "jax_backend": "cpu",
    "jax_devices": [
      "TFRT_CPU_0"
    ],
    "physical_cores": 20,
    "logical_cores": 20
  },
  "WDM": {
    "root": "/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets",
    "commit": "a927f4f1e9fde6e2b9925a008040b6572ef4d35e",
    "dirty": false
  },
  "response": {
    "root": "/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse",
    "commit": "14cb626cad16cfaf715594e51eb4e668431a5ca2",
    "dirty": true
  }
}
Initial memory: {'rss_bytes': 893546496, 'peak_rss_bytes': 891944960, 'rss_gib': 0.8321800231933594, 'peak_rss_gib': 0.8306884765625}


/tmp/ipykernel_535719/1480319648.py:16: UserWarning: WDM commit a927f4f1e9fde6e2b9925a008040b6572ef4d35e does not match the expected prefix 'c588649f5'.
  warnings.warn(message)
/tmp/ipykernel_535719/1480319648.py:22: UserWarning: At least one benchmark checkout is dirty; the JSON record will preserve this fact.
  warnings.warn(message)


## 3. Reuse or create the full-year waveform cache

The paths deliberately match notebook 03 so the existing `calx038` cache is reused. Set the reuse flags to `False` after changing waveform generation, sampling, padding, or WDM tiling.

In [5]:
CACHE_DIR = WDM_LISA_ROOT / "examples"
WAVEFORM_CACHE = CACHE_DIR / "_local_wdm_03_waveform_cache.npz"
WDM_CACHE = CACHE_DIR / "_local_wdm_03_wdm_coefficients.npz"
CACHE_METADATA = CACHE_DIR / "_local_wdm_03_cache_metadata.json"

if REUSE_WAVEFORM_CACHE and WAVEFORM_CACHE.exists():
    with np.load(WAVEFORM_CACHE, allow_pickle=False) as cached:
        times_pad = np.asarray(cached["times_pad"])
        h_plus = np.asarray(cached["h_plus"])
        h_cross = np.asarray(cached["h_cross"])

    waveform_generation_s = 0.0
    print("Reused:", WAVEFORM_CACHE)

else:
    td_gen = GenerateEMRIWaveform(
        "FastKerrEccentricEquatorialFlux",
        sum_kwargs={"pad_output": True, "odd_len": True},
        return_list=True,
        force_backend="cpu",
    )

    model_kwargs = {
        "td_gen": td_gen,
        "m1": 1.0e6,
        "m2": 10.0,
        "a": 0.9,
        "p0": 8.5,
        "e0": 0.3,
        "x0": 1.0,
        "dist": 1.0,
        "theta": THETA,
        "phi": PHI,
        "qK": np.pi / 3,
        "phiK": np.pi / 3,
        "Phi_phi0": np.pi / 3,
        "Phi_theta0": 0.0,
        "Phi_r0": np.pi / 3,
        "eps": 1.0e-2,
    }

    waveform_payload, waveform_generation_s = timed_call(
        lambda: generate_windowed_padded_waveform(
            waveform_model="emri",
            t0=t0,
            dt=DT,
            N=N_UNPADDED,
            model_kwargs=model_kwargs,
            window_function="tukey",
            window_kwargs={
                "alpha": WINDOW_ALPHA,
                "tukey_factor": TUKEY_FACTOR,
            },
            wdm_Nf=NF,
            return_padding_info=True,
        )
    )

    times_pad, h_plus, h_cross, _padding = waveform_payload

    np.savez(
        WAVEFORM_CACHE,
        times_pad=times_pad,
        h_plus=h_plus,
        h_cross=h_cross,
    )

    print("Created:", WAVEFORM_CACHE)

if not (times_pad.shape == h_plus.shape == h_cross.shape):
    raise ValueError(
        "Cached time and polarization arrays must have identical shapes: "
        f"times={times_pad.shape}, "
        f"plus={h_plus.shape}, "
        f"cross={h_cross.shape}."
    )

if times_pad.size != EXPECTED_PADDED_N:
    raise ValueError(
        f"Expected {EXPECTED_PADDED_N} samples after padding "
        f"({N_UNPADDED} input + {WINDOW_PAD_PER_SIDE} per side), "
        f"found {times_pad.size}."
    )

if times_pad.size < 2 or not np.isclose(
    times_pad[1] - times_pad[0],
    DT,
):
    raise ValueError(
        f"Cached waveform sampling does not match dt={DT} s."
    )

print("Input samples:", N_UNPADDED)
print("Padding per side:", WINDOW_PAD_PER_SIDE)
print("Padded samples:", times_pad.size)
print("Expected WDM Nt:", EXPECTED_WDM_NT)
print("Waveform generation [s]:", waveform_generation_s)
print(
    "Waveform host memory [GiB]:",
    (times_pad.nbytes + h_plus.nbytes + h_cross.nbytes) / 1024**3,
)
print("Memory:", memory_snapshot())

Reused: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_waveform_cache.npz
Input samples: 3200000
Padding per side: 32000
Padded samples: 3264000
Expected WDM Nt: 32640
Waveform generation [s]: 0.0
Waveform host memory [GiB]: 0.07295608520507812
Memory: {'rss_bytes': 972677120, 'peak_rss_bytes': 971235328, 'rss_gib': 0.9058761596679688, 'peak_rss_gib': 0.9045333862304688}


## 4. Build the WDM grid and frozen production plan

In [6]:
grid_context, grid_build_s = timed_call(
    lambda: _build_wdm_dT_grid(
        times_pad,
        Nf=NF,
        d=D_WDM,
        A_frac=A_FRAC,
        q=Q_WDM,
        calc_m0=CALC_M0,
    )
)

def build_one_arm_plan(shift_overrides=None):
    shift_kwargs = {**BASE_SHIFT_KWARGS, **(shift_overrides or {})}
    return OneArmWDMStaticPlan.build_from_context(
        grid_context,
        orbits,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=shift_kwargs,
    )

baseline_plan, baseline_plan_build_s = timed_call(build_one_arm_plan)
expected_shape = (grid_context["wdm"].Nt, grid_context["wdm"].Nf)

print("WDM shape:", expected_shape)
print("Grid build [s]:", grid_build_s)
print("Static plan build [s]:", baseline_plan_build_s)
print("Interpolation table [MiB]:", baseline_plan.kernel_memory_bytes / 1024**2)
print("Memory:", memory_snapshot())


WDM shape: (32640, 100)
Grid build [s]: 2.8083534789984697
Static plan build [s]: 0.7583272720003151
Interpolation table [MiB]: 1.6015625
Memory: {'rss_bytes': 1424674816, 'peak_rss_bytes': 1871585280, 'rss_gib': 1.3268318176269531, 'peak_rss_gib': 1.7430496215820312}


## 5. Reuse or create WDM coefficient cache, then benchmark DWTs

The shift sweeps use one fixed pair of WDM coefficient arrays. Complete A1 below recomputes the transforms inside the timed callable.

In [7]:
def sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )

def parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )

dwt_results = {}
if RUN_DWT_BENCHMARK:
    sequential_context, sequential_row = benchmark_candidate(
        "sequential_dwt",
        sequential_dwt,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["repeats"],
        m_start=1,
    )
    parallel_context, parallel_row = benchmark_candidate(
        "parallel_dwt",
        parallel_dwt,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["repeats"],
        m_start=1,
    )
    dwt_comparison = {
        "plus": compare_arrays(parallel_context["w_t_plus"], sequential_context["w_t_plus"], m_start=1),
        "cross": compare_arrays(parallel_context["w_t_cross"], sequential_context["w_t_cross"], m_start=1),
    }
    dwt_results = {
        "sequential": sequential_row,
        "parallel": parallel_row,
        "comparison": dwt_comparison,
    }
    print(pd.DataFrame([sequential_row, parallel_row])[["name", "first_call_s", "median_s", "iqr_s"]])
    print("DWT comparison:", dwt_comparison)
else:
    parallel_context = parallel_dwt()
    block_result(parallel_context)

cache_is_compatible = False
if REUSE_WDM_CACHE and WDM_CACHE.exists():
    with np.load(WDM_CACHE, allow_pickle=False) as cached:
        cached_plus = np.asarray(cached["w_plus"])
        cached_cross = np.asarray(cached["w_cross"])
        cached_nf = int(np.asarray(cached["Nf"]).item())
        cached_calc_m0 = bool(np.asarray(cached["calc_m0"]).item())
    cache_is_compatible = (
        cached_plus.shape == expected_shape
        and cached_cross.shape == expected_shape
        and cached_nf == NF
        and cached_calc_m0 == CALC_M0
    )

if cache_is_compatible:
    w_plus_host = cached_plus
    w_cross_host = cached_cross
    print("Reused:", WDM_CACHE)
else:
    w_plus_host = np.asarray(parallel_context["w_t_plus"])
    w_cross_host = np.asarray(parallel_context["w_t_cross"])
    np.savez(
        WDM_CACHE,
        times_pad=times_pad,
        w_plus=w_plus_host,
        w_cross=w_cross_host,
        Nf=np.asarray(NF),
        q_wdm=np.asarray(Q_WDM),
        d_wdm=np.asarray(D_WDM),
        A_frac=np.asarray(A_FRAC),
        calc_m0=np.asarray(CALC_M0),
    )
    print("Created:", WDM_CACHE)

if w_plus_host.shape != expected_shape or w_cross_host.shape != expected_shape:
    raise ValueError("Cached WDM coefficient shape does not match the active grid.")

w_plus = jnp.asarray(w_plus_host)
w_cross = jnp.asarray(w_cross_host)
block_result((w_plus, w_cross))
print("Coefficient dtype:", w_plus.dtype)
print("Coefficient device memory [MiB]:", (w_plus.nbytes + w_cross.nbytes) / 1024**2)


             name  first_call_s  median_s     iqr_s
0  sequential_dwt      0.655261  0.208024  0.004892
1    parallel_dwt      0.122659  0.118974  0.003779
DWT comparison: {'plus': {'max_abs': 0.0, 'rel_l2': 0.0}, 'cross': {'max_abs': 0.0, 'rel_l2': 0.0}}
Reused: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_wdm_coefficients.npz
Coefficient dtype: float64
Coefficient device memory [MiB]: 49.8046875


## 6. Freeze the current six-link reference

Every later candidate is compared with this output. Timing uses the currently validated groups-of-two/three-worker schedule; the first call is reported separately from steady-state samples.

In [8]:
def apply_plan(
    plan,
    *,
    mode="runtime_sky",
    group_size=6,
    workers=1,
    plus=w_plus,
    cross=w_cross,
):
    del mode, group_size, workers
    return plan.apply_device(
        plus,
        cross,
        THETA,
        PHI,
    )

reference_links, baseline_shift_row = benchmark_candidate(
    "current_runtime_sky",
    lambda: apply_plan(baseline_plan),
    warmups=PROFILE["warmups"],
    repeats=PROFILE["repeats"],
    m_start=1,
    metadata={
        "row_chunk_size": BASE_ROW_CHUNK_SIZE,
        "lag_block_size": BASE_LAG_BLOCK_SIZE,
        "execution_mode": "runtime_sky",
    },
)

print(json.dumps(baseline_shift_row, indent=2))


{
  "name": "current_runtime_sky",
  "first_call_s": 1.309108223002113,
  "median_s": 0.767068223998649,
  "q25_s": 0.7621367560022918,
  "q75_s": 0.7792241259994626,
  "iqr_s": 0.017087369997170754,
  "minimum_s": 0.745453137998993,
  "maximum_s": 0.7867608160013333,
  "all_s": [
    0.767068223998649,
    0.7534821499939426,
    0.7711889150014031,
    0.7745459290017607,
    0.7839023229971644,
    0.7867608160013333,
    0.745453137998993,
    0.7650354940051329,
    0.7862208139995346,
    0.7592380179994507,
    0.7657277609978337
  ],
  "max_abs": 0.0,
  "rel_l2": 0.0,
  "per_link": {},
  "rss_before_gib": 2.20068359375,
  "rss_after_gib": 2.5218772888183594,
  "process_peak_rss_gib": 3.1815032958984375,
  "row_chunk_size": 32640,
  "lag_block_size": 17,
  "execution_mode": "runtime_sky"
}


## 7. Common plan-candidate runner

Each temporary plan is built, benchmarked, compared, and released before the next candidate. `plan_build_s` is recorded but excluded from repeated-application timing.

In [9]:
def evaluate_plan_candidate(
    *,
    name,
    shift_overrides,
    mode="runtime_sky",
    group_size=6,
    workers=1,
):
    candidate_plan = None
    candidate_output = None
    try:
        candidate_plan, plan_build_s = timed_call(
            lambda: build_one_arm_plan(shift_overrides)
        )
        candidate_output, row = benchmark_candidate(
            name,
            lambda: apply_plan(
                candidate_plan,
                mode=mode,
                group_size=group_size,
                workers=workers,
            ),
            reference=reference_links,
            warmups=PROFILE["warmups"],
            repeats=PROFILE["repeats"],
            m_start=1,
            metadata={
                "plan_build_s": plan_build_s,
                "plan_bytes": candidate_plan.kernel_memory_bytes,
                "row_chunk_size": int(shift_overrides.get("row_chunk_size", BASE_ROW_CHUNK_SIZE)),
                "lag_block_size": int(shift_overrides.get("lag_block_size", BASE_LAG_BLOCK_SIZE)),
                "group_size": int(group_size),
                "workers": int(workers),
                "execution_mode": mode,
                "shift_overrides": dict(shift_overrides),
            },
        )
        row["passes_numerics"] = numerical_gate(
            row,
            rel_l2_tolerance=REL_L2_TOLERANCE,
            max_abs_tolerance=MAX_ABS_TOLERANCE,
        )
        return row
    finally:
        del candidate_output
        if candidate_plan is not None:
            candidate_plan.clear_device_cache()
        del candidate_plan
        gc.collect()


## 8. Stage 1 — row-chunk sweep

Lag blocking and scheduling are held fixed. The materialized six-job path isolates the compiled assembly kernel from Python link scheduling.

In [10]:
row_sweep = []
for row_size in PROFILE["row_sizes"]:
    print(f"Row chunk {row_size:,}")
    row_sweep.append(
        evaluate_plan_candidate(
            name=f"row_{row_size}",
            shift_overrides={
                "row_chunk_size": int(row_size),
                "lag_block_size": BASE_LAG_BLOCK_SIZE,
            },
        )
    )

best_row_result = fastest_passing(
    row_sweep,
    rel_l2_tolerance=REL_L2_TOLERANCE,
    max_abs_tolerance=MAX_ABS_TOLERANCE,
)
BEST_ROW_CHUNK_SIZE = int(best_row_result["row_chunk_size"])

row_sweep_table = pd.DataFrame(row_sweep)
display(
    row_sweep_table[
        [
            "name",
            "row_chunk_size",
            "first_call_s",
            "median_s",
            "iqr_s",
            "max_abs",
            "rel_l2",
            "passes_numerics",
            "rss_after_gib",
        ]
    ].sort_values("median_s")
)
print("Selected row_chunk_size:", BEST_ROW_CHUNK_SIZE)


Row chunk 4,096
Row chunk 8,192
Row chunk 16,000
Row chunk 24,000
Row chunk 32,000
Row chunk 32,640


,name,row_chunk_size,first_call_s,median_s,iqr_s,max_abs,rel_l2,passes_numerics,rss_after_gib
5,row_32640,32640,0.770250,0.766585,0.011801,0.0,0.0,True,2.826439
1,row_8192,8192,0.961618,0.856491,0.008839,0.0,0.0,True,2.792999
0,row_4096,4096,0.989782,0.888217,0.013666,0.0,0.0,True,2.749462
3,row_24000,24000,1.044979,0.966194,0.015392,0.0,0.0,True,2.820747
2,row_16000,16000,1.035800,0.971729,0.011595,0.0,0.0,True,2.805214
4,row_32000,32000,1.257652,1.139355,0.008354,0.0,0.0,True,2.813026


Selected row_chunk_size: 32640


## 9. Stage 2 — lag-block sweep

This changes only how the same 51 lags are blocked. It does **not** change $L_{\rm trunc}$.

In [11]:
lag_sweep = []
for lag_size in PROFILE["lag_sizes"]:
    print(f"Lag block {lag_size}")
    lag_sweep.append(
        evaluate_plan_candidate(
            name=f"row_{BEST_ROW_CHUNK_SIZE}_lag_{lag_size}",
            shift_overrides={
                "row_chunk_size": BEST_ROW_CHUNK_SIZE,
                "lag_block_size": int(lag_size),
            },
        )
    )

best_lag_result = fastest_passing(
    lag_sweep,
    rel_l2_tolerance=REL_L2_TOLERANCE,
    max_abs_tolerance=MAX_ABS_TOLERANCE,
)
BEST_LAG_BLOCK_SIZE = int(best_lag_result["lag_block_size"])

lag_sweep_table = pd.DataFrame(lag_sweep)
display(
    lag_sweep_table[
        [
            "name",
            "row_chunk_size",
            "lag_block_size",
            "first_call_s",
            "median_s",
            "iqr_s",
            "max_abs",
            "rel_l2",
            "passes_numerics",
        ]
    ].sort_values("median_s")
)
print("Selected lag_block_size:", BEST_LAG_BLOCK_SIZE)


Lag block 3
Lag block 8
Lag block 16
Lag block 17
Lag block 25
Lag block 51


,name,row_chunk_size,lag_block_size,first_call_s,median_s,iqr_s,max_abs,rel_l2,passes_numerics
1,row_32640_lag_8,32640,8,0.967985,0.824359,0.019994,1.734723e-18,1.054040e-07,True
2,row_32640_lag_16,32640,16,0.928085,0.858488,0.017392,1.734723e-18,9.532769e-08,True
0,row_32640_lag_3,32640,3,0.968156,0.871529,0.010342,1.734723e-18,1.234052e-07,True
3,row_32640_lag_17,32640,17,0.775104,0.913297,0.075541,0.000000e+00,0.000000e+00,True
4,row_32640_lag_25,32640,25,1.146385,0.927956,0.030166,1.951564e-18,1.368474e-07,True
5,row_32640_lag_51,32640,51,2.876925,2.707620,0.041033,2.168404e-18,1.170210e-07,True


Selected lag_block_size: 8


## 10. Stage 3 — link-schedule sweep

The winning row/lag configuration is built once. This stage chooses how the six directed links are divided across CPU workers.

In [12]:
tuned_shift_overrides = {
    "row_chunk_size": BEST_ROW_CHUNK_SIZE,
    "lag_block_size": BEST_LAG_BLOCK_SIZE,
}
tuned_matches_baseline = (
    BEST_ROW_CHUNK_SIZE == BASE_ROW_CHUNK_SIZE
    and BEST_LAG_BLOCK_SIZE == BASE_LAG_BLOCK_SIZE
)
if tuned_matches_baseline:
    tuned_plan = baseline_plan
    tuned_plan_build_s = baseline_plan_build_s
else:
    tuned_plan, tuned_plan_build_s = timed_call(
        lambda: build_one_arm_plan(tuned_shift_overrides)
    )

schedule_sweep = []
for schedule in SCHEDULES:
    print(schedule["name"])
    schedule_output, schedule_row = benchmark_candidate(
        schedule["name"],
        lambda schedule=schedule: apply_plan(
            tuned_plan,
            mode=schedule["mode"],
            group_size=schedule["group_size"],
            workers=schedule["workers"],
        ),
        reference=reference_links,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["repeats"],
        m_start=1,
        metadata={
            "row_chunk_size": BEST_ROW_CHUNK_SIZE,
            "lag_block_size": BEST_LAG_BLOCK_SIZE,
            "group_size": schedule["group_size"],
            "workers": schedule["workers"],
            "execution_mode": schedule["mode"],
        },
    )
    schedule_row["passes_numerics"] = numerical_gate(
        schedule_row,
        rel_l2_tolerance=REL_L2_TOLERANCE,
        max_abs_tolerance=MAX_ABS_TOLERANCE,
    )
    schedule_row["speedup_vs_current_selected"] = (
        baseline_shift_row["median_s"] / schedule_row["median_s"]
    )
    schedule_sweep.append(schedule_row)
    del schedule_output
    gc.collect()

best_schedule = fastest_passing(
    schedule_sweep,
    rel_l2_tolerance=REL_L2_TOLERANCE,
    max_abs_tolerance=MAX_ABS_TOLERANCE,
)

schedule_sweep_table = pd.DataFrame(schedule_sweep)
display(
    schedule_sweep_table[
        [
            "name",
            "group_size",
            "workers",
            "first_call_s",
            "median_s",
            "iqr_s",
            "speedup_vs_current_selected",
            "max_abs",
            "rel_l2",
            "passes_numerics",
        ]
    ].sort_values("median_s")
)
print("Selected schedule:", best_schedule["name"])


runtime_sky


,name,group_size,workers,first_call_s,median_s,iqr_s,speedup_vs_current_selected,max_abs,rel_l2,passes_numerics
0,runtime_sky,6,1,0.799452,0.8272,0.010552,0.927307,1.734723e-18,1.054040e-07,True


Selected schedule: runtime_sky


## 11. Optional new-backend experiments

After implementing reordered kernels, cached phases, factored phases, a real split kernel, compact interpolation, boundary specialisation, or frequency tiling, add one configuration in `CUSTOM_EXPERIMENTS`. Keep each refactor isolated until it passes this notebook.

In [13]:
custom_results = []
for experiment in CUSTOM_EXPERIMENTS:
    custom_results.append(
        evaluate_plan_candidate(
            name=experiment["name"],
            shift_overrides=experiment["shift_overrides"],
            mode=experiment.get("mode", "materialized"),
            group_size=experiment.get("group_size", 6),
            workers=experiment.get("workers", 1),
        )
    )

if custom_results:
    custom_table = pd.DataFrame(custom_results)
    display(
        custom_table[
            [
                "name",
                "median_s",
                "iqr_s",
                "max_abs",
                "rel_l2",
                "passes_numerics",
                "plan_bytes",
            ]
        ].sort_values("median_s")
    )
else:
    print("No custom backend experiments configured.")


No custom backend experiments configured.


## 12. Complete A1 confirmation

This is the relevant end-to-end code benchmark: two parallel DWTs followed by all six one-arm responses. Scientific mismatch against the time-domain response remains a separate validation step.

In [14]:
a1_results = {}
if RUN_COMPLETE_A1:
    def current_a1():
        context = parallel_dwt()
        return apply_plan(
            baseline_plan,
            plus=context["w_t_plus"],
            cross=context["w_t_cross"],
        )

    def tuned_a1():
        context = parallel_dwt()
        return apply_plan(
            tuned_plan,
            plus=context["w_t_plus"],
            cross=context["w_t_cross"],
        )

    current_a1_output, current_a1_row = benchmark_candidate(
        "current_a1",
        current_a1,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["a1_repeats"],
        m_start=1,
    )
    tuned_a1_output, tuned_a1_row = benchmark_candidate(
        "tuned_a1",
        tuned_a1,
        reference=current_a1_output,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["a1_repeats"],
        m_start=1,
    )
    a1_results = {
        "current": current_a1_row,
        "tuned": tuned_a1_row,
    }
    print(json.dumps(a1_results, indent=2))


{
  "current": {
    "name": "current_a1",
    "first_call_s": 0.9331284970030538,
    "median_s": 0.8942342949958402,
    "q25_s": 0.889647664000222,
    "q75_s": 0.906479343000683,
    "iqr_s": 0.01683167900046101,
    "minimum_s": 0.8734680950001348,
    "maximum_s": 0.9389633230020991,
    "all_s": [
      0.9144446040008916,
      0.906479343000683,
      0.8916411449972657,
      0.9389633230020991,
      0.8866703179955948,
      0.8734680950001348,
      0.8942342949958402,
      0.889647664000222,
      0.8945805559997098
    ],
    "max_abs": 0.0,
    "rel_l2": 0.0,
    "per_link": {},
    "rss_before_gib": 2.86639404296875,
    "rss_after_gib": 3.020793914794922,
    "process_peak_rss_gib": 10.827461242675781
  },
  "tuned": {
    "name": "tuned_a1",
    "first_call_s": 0.9576103040017188,
    "median_s": 0.9648602899978869,
    "q25_s": 0.9634561150014633,
    "q75_s": 0.9743318570035626,
    "iqr_s": 0.010875742002099287,
    "minimum_s": 0.9573347870027646,
    "maximum_s

## 13. Decision gate and export

A tuning winner is recorded even if the gain is small, but it is marked `meaningful_speedup=False` unless it is at least 5% faster than the frozen selected baseline. New code refactors should generally be retained only if they clear that threshold or deliver a material memory reduction.

In [15]:
selected_shift_speedup = (
    baseline_shift_row["median_s"] / best_schedule["median_s"]
)
meaningful_speedup = selected_shift_speedup >= MIN_MEANINGFUL_SPEEDUP

selection = {
    "row_chunk_size": BEST_ROW_CHUNK_SIZE,
    "lag_block_size": BEST_LAG_BLOCK_SIZE,
    "schedule_name": "runtime_sky",
    "execution_mode": "runtime_sky",
    "group_size": 6,
    "workers": 1,
    "shift_speedup_vs_current_selected": float(selected_shift_speedup),
    "meaningful_speedup": bool(meaningful_speedup),
    "passes_numerics": bool(best_schedule["passes_numerics"]),
}

results = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "host": HOST,
    "source": {"WDM": WDM_STATE, "WDM_LISAresponse": RESPONSE_STATE},
    "provenance_ok": provenance_ok,
    "configuration": {
        "N": N_UNPADDED,
        "N_unpadded": N_UNPADDED,
        "N_padded": int(times_pad.size),
        "window_alpha": WINDOW_ALPHA,
        "tukey_factor": TUKEY_FACTOR,
        "window_pad_per_side": WINDOW_PAD_PER_SIDE,
        "dt": DT,
        "Nf": NF,
        "q": Q_WDM,
        "d": D_WDM,
        "A_frac": A_FRAC,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "baseline_shift_kwargs": BASE_SHIFT_KWARGS,
        "rel_l2_tolerance": REL_L2_TOLERANCE,
        "max_abs_tolerance": MAX_ABS_TOLERANCE,
    },
    "cache": {
        "waveform": WAVEFORM_CACHE,
        "wdm_coefficients": WDM_CACHE,
    },
    "setup_s": {
        "waveform_generation": waveform_generation_s,
        "grid": grid_build_s,
        "baseline_plan": baseline_plan_build_s,
        "tuned_plan": tuned_plan_build_s,
    },
    "dwt": dwt_results,
    "current_selected_shift": baseline_shift_row,
    "row_sweep": row_sweep,
    "lag_sweep": lag_sweep,
    "schedule_sweep": schedule_sweep,
    "custom_experiments": custom_results,
    "complete_a1": a1_results,
    "selection": selection,
    "final_memory": memory_snapshot(),
}

print(json.dumps(selection, indent=2))

if SAVE_RESULTS:
    RESULTS_DIR = BENCHMARK_DIR / "runtime_results"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    node = (HOST.get("node") or "unknown").replace("/", "_")
    result_stem = f"dense_runtime_{node}_{RUN_MODE}_{timestamp}"
    json_path = save_json(RESULTS_DIR / f"{result_stem}.json", results)

    flat_rows = []
    for stage, rows in (
        ("row", row_sweep),
        ("lag", lag_sweep),
        ("schedule", schedule_sweep),
        ("custom", custom_results),
    ):
        for row in rows:
            flat_rows.append(
                {
                    "stage": stage,
                    **{
                        key: value
                        for key, value in row.items()
                        if key not in {"all_s", "per_link", "shift_overrides"}
                    },
                }
            )
    csv_path = RESULTS_DIR / f"{result_stem}.csv"
    pd.DataFrame(flat_rows).to_csv(csv_path, index=False)
    print("Saved:", json_path)
    print("Saved:", csv_path)


{
  "row_chunk_size": 32640,
  "lag_block_size": 8,
  "schedule_name": "runtime_sky",
  "execution_mode": "runtime_sky",
  "group_size": 6,
  "workers": 1,
  "shift_speedup_vs_current_selected": 0.9273066072955303,
  "meaningful_speedup": false,
  "passes_numerics": true
}
Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/testing_optimisation/runtime_results/dense_runtime_calx038.ast.cam.ac.uk_full_20260727_141616.json
Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/testing_optimisation/runtime_results/dense_runtime_calx038.ast.cam.ac.uk_full_20260727_141616.csv


## 14. Promotion checklist

Before changing the production defaults:

1. Set `RUN_MODE="full"` and restart the kernel.
2. Run the current baseline first, then the candidate; restart and reverse the order.
3. Confirm that the medians/IQRs agree across the two fresh processes.
4. Require the numerical gate, complete-A1 gate, and package tests to pass.
5. Run the winning implementation through the existing full response/mismatch notebook.
6. Commit one optimisation at a time and make the winner the next frozen baseline.

Do not infer isolated candidate peak memory from the cumulative process high-water mark. If memory is the claim, rerun only the baseline or only the candidate in a fresh process.